In [0]:
dbutils.widgets.text("login", "")
dbutils.widgets.text("catalog", "")

In [0]:
login = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("catalog")
bronze_schema = f"{login}_bronze"
dataset = "genai_llm_usage_dataset_1000"

volume_path = f"/Volumes/{catalog}/{bronze_schema}/data"

checkpoint_path = f"/Volumes/{catalog}/{bronze_schema}/checkpoints/{dataset}/checkpoint"
schema_path = f"{checkpoint_path}/schema"
target_table = f"{catalog}.{bronze_schema}.{dataset}_bronze"

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

Auto Loader Idempotency

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(volume_path)
)

In [0]:
bronze_df = (
    source_df
    .withColumn("source_filename", col("_metadata.file_name"))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("ingested_date", current_date())
)

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
display(dbutils.fs.ls(checkpoint_path))

In [0]:
print(query.status)

In [0]:
display(spark.table(target_table))

In [0]:
spark.table(target_table).count()

MANUAL IDEMPOTENCY USING NOT EXISTS

```
source_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(volume_path)
)

bronze_df = (
    source_df
    .withColumn("source_filename", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("ingested_date", current_date())
)

target_exists = spark.catalog.tableExists(target_table)

if target_exists:
    loaded_files_df = (
        spark.table(target_table).select("source_filename").distinct()
    )

    new_data_df = bronze_df.join(
        loaded_files_df,
        on = "source_filename", 
        how="left_anti"
    )
else:
    new_data_df = bronze_df

display(new_data_df)
```